### Imports

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import sys
from pathlib import Path

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))


from src.paths import *
from src.data_validation import validate_grid
from src.graph_utils import build_grid_graph

### Load the datasets

In [3]:
substations_path = GIS / "grid_substations.csv"
lines_path = GIS / "transmission_lines.csv"
if substations_path.is_file() and lines_path.is_file():
    substations = pd.read_csv(substations_path)
    transmission_lines = pd.read_csv(lines_path)
    validate_grid(substations, transmission_lines)
else:
    print("Grid graph construction skipped: completed GIS inputs are missing.")
    print(f"Provide {substations_path.name} and {lines_path.name} in {GIS}.")
    print("Start from datasets/templates/grid_substations_template.csv and transmission_lines_template.csv.")
    substations = pd.DataFrame(columns=["bus_id", "name", "latitude", "longitude", "voltage_kv"])
    transmission_lines = pd.DataFrame(columns=["from_bus", "to_bus", "length_km"])

### Construct the graph

In [5]:
G = build_grid_graph(
    substations,
    transmission_lines
)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

print("Is connected:", nx.is_connected(G) if G.number_of_nodes() else False)
print("Connected components:", nx.number_connected_components(G))

Number of nodes: 30
Number of edges: 39
Is connected: True
Connected components: 1


### Examine graph properties

In [7]:
print("Graph density:", nx.density(G))

if G.number_of_nodes():
    print("Average degree:", sum(dict(G.degree()).values()) / G.number_of_nodes())
    print("Largest connected component:", len(max(nx.connected_components(G), key=len)))
else:
    print("Network metrics unavailable: no substations were loaded.")

Graph density: 0.0896551724137931
Average degree: 2.6
Largest connected component: 30


### Visualize the transmission graph

In [9]:
if G.number_of_nodes() == 0:
    plt.figure(figsize=(12, 6))
    plt.axis("off")
    plt.text(
        0.5, 0.5,
        "Transmission Network Graph is Empty\n\n"
        "Missing GIS input datasets:\n"
        f"- {GIS / 'grid_substations.csv'}\n"
        f"- {GIS / 'transmission_lines.csv'}\n\n"
        "Populate these CSV files in datasets/raw/gis to visualize the network.",
        ha="center", va="center", fontsize=12,
        bbox={"facecolor": "white", "edgecolor": "gray", "boxstyle": "round,pad=1"}
    )
    plt.title("Sri Lanka Electricity Transmission Network (No Data Loaded)")
    plt.tight_layout()
    plt.savefig(GRAPHS / "transmission_network.png", dpi=300)
    plt.show()
else:
    plt.figure(figsize=(14, 10))
    has_geo = all("longitude" in G.nodes[n] and "latitude" in G.nodes[n] for n in G.nodes)
    if has_geo:
        pos = {n: (float(G.nodes[n]["longitude"]), float(G.nodes[n]["latitude"])) for n in G.nodes}
    else:
        pos = nx.spring_layout(G, seed=42, weight="length_km")

    nodes_220 = [n for n in G.nodes if float(G.nodes[n].get("voltage_kv", 0)) >= 220]
    nodes_132 = [n for n in G.nodes if float(G.nodes[n].get("voltage_kv", 0)) < 220]

    if nodes_220:
        nx.draw_networkx_nodes(G, pos, nodelist=nodes_220, node_size=180, node_color="#d62728", label="220 kV Substation")
    if nodes_132:
        nx.draw_networkx_nodes(G, pos, nodelist=nodes_132, node_size=110, node_color="#1f77b4", label="132 kV Substation")

    nx.draw_networkx_edges(G, pos, width=1.2, alpha=0.6, edge_color="gray")
    nx.draw_networkx_labels(G, pos, font_size=7, font_family="sans-serif", font_weight="bold")

    plt.title("Sri Lanka Electricity Transmission Network (Grid Topology)", fontsize=14)
    if has_geo:
        plt.xlabel("Longitude (°E)", fontsize=11)
        plt.ylabel("Latitude (°N)", fontsize=11)
        plt.grid(True, linestyle="--", alpha=0.4)
    else:
        plt.axis("off")
    plt.legend(loc="upper left", frameon=True)
    plt.tight_layout()

    plt.savefig(GRAPHS / "transmission_network.png", dpi=300)
    plt.show()


### Export graph information

In [11]:
if G.number_of_nodes() > 0:
    nodes = pd.DataFrame([
        {
            "bus_id": node,
            **attributes
        }
        for node, attributes in G.nodes(data=True)
    ])
else:
    nodes = pd.DataFrame(columns=["bus_id", "name", "latitude", "longitude", "voltage_kv"])

if G.number_of_edges() > 0:
    edges = pd.DataFrame([
        {
            "from_bus": u,
            "to_bus": v,
            **attributes
        }
        for u, v, attributes in G.edges(data=True)
    ])
else:
    edges = pd.DataFrame(columns=["from_bus", "to_bus", "length_km"])

nodes.to_csv(
    PROCESSED / "grid_nodes.csv",
    index=False
)

edges.to_csv(
    PROCESSED / "grid_edges.csv",
    index=False
)
